# Pendulum example

- a simple pendulum (a mass on a string) can be used to measure the gravitational acceleration g
- such a measurement was performed by students in a lab class and the results were recorded and stored in a CSV (comma separated values) file
- the measurements are a mix of "first pendulum" (large uncertainty) and "best pendulum" (smaller uncertainty) measurements

- __First step: Let's look at the raw data__

In [ ]:
# open the text file containing data

f = open("./pendulumData.csv","r")

# Use a while loop to read the data line-by-line until the end of file
flag = True
i = 0
while flag:
    s = f.readline()
    if s:
        print(i,s)
        i = i + 1
    else:
        # reached end of file; stop while loop
        flag = False
        
# close the data file
f.close()


__Second step: Read in the raw data, make a histogram, plot the data and examine it__

In [ ]:
#### import numpy as np
import matplotlib.pylab as plt
import numpy as np

# open a text file containing data

f = open("../Data/pendulumData.csv","r")

#read one line of data and discard header line

s = f.readline()

# define empty array to store data values
g = []

#the while loop will run until the end of file or i == nevent
flag = True
i = 0
while flag:
    s = f.readline()
    if s:
#        print(i,s)
        ss = s.split(",")
        g.append(float(ss[1]))
        i = i + 1
    else:
        # reached end of file; stop while loop
        flag = False
        
print("# of measurements read is",i)
f.close()

#plt.subplot(110)


nbins = 100

plt.figure(figsize=(10,10))
counts, edges, something  = plt.hist(g,nbins, color = 'r')

g_max = np.amax(g)
g_ave = np.average(g)
g_std = np.std(g)
g_uncertainty = g_std/np.sqrt(i)

par = [g_max,g_ave,g_std]
g_max = np.amax(counts)
g_ave = np.average(g)
g_std = np.std(g)
g_uncertainty = g_std/np.sqrt(i)
print("Gaussian parameters:",par)
print("<g> = ",g_ave,"std(g) = ",g_std)
print("uncertainty sigma/sqrt(N) on <g> =",g_uncertainty)

plt.xlabel(r'g[m/$s^2$]', fontsize = 15)
plt.ylabel(r'#of measurements per bin', fontsize = 15)


- most of the data are close to 10m/s^2, but one data point is at 980 m/s^2
- data points like this are called outliers
- what to do with data like this?
- Do not just throw them away, but think how this might have happened
- is there an explanation? Was this a mistake or is this a big discovery?
- in this case, it is __likely__ that someone reported the result in cm/s^2 instead of m/s^2, making a factor of 100 difference
- let's do this again, but clean up the data before plotting

In [ ]:
#### import numpy as np
import matplotlib.pylab as plt
import numpy as np

# example of function definition
def gauss(x, a, x0, sigma):
   return a*np.exp(-(x-x0)**2/(2*(sigma**2)))

# open a text file containing data

f = open("../Data/pendulumData.csv","r")

#read one line of data and discard

s = f.readline()

# define empty array to store data values
g = []


#the while loop will run until the end of file or i == nevent
flag = True
i = 0
while flag:
    s = f.readline()
    if s:
#        print(i,s)
        ss = s.split(",")
        x = float(ss[1])
        if(x > 500.0):
            x = x/100.0
        g.append(x)
        i = i + 1
    else:
        # reached end of file; stop while loop
        flag = False
        
print("# of measurements read is",i)
f.close()

nbins = 100

#range of histogram
xmin = 6
xmax = 16

plt.figure(figsize=(10,10))
counts, edges, something  = plt.hist(g,nbins, color = 'r',range = [xmin,xmax])


plt.xlabel(r'g[m/$s^2$]', fontsize = 15)
plt.ylabel(r'#of measurements per bin', fontsize = 15)


- Ok, that looks better
- now we can look at the important statistical properties of the distribution

In [ ]:
g_max = np.amax(g)
g_ave = np.average(g)
g_std = np.std(g)
g_uncertainty = g_std/np.sqrt(i)

print("<g> = ",g_ave,"std(g) = ",g_std)
print("uncertainty sigma/sqrt(N) on <g> =",g_uncertainty)


- we can do better by just plotting the __significant digits__
- out uncertainty is about 0.04. So digits much smaller than this are just noise with no information

In [ ]:
print("Final result:")
print(f'<g> = {g_ave:.3f}',f'std(g) = {g_std:.3f}')
print(f'uncertainty sigma/sqrt(N) on <g> = {g_uncertainty:.3f}')

In [ ]:
#Import the curve_fit function from scipy

from scipy.optimize import curve_fit

# Define a Gaussian function
def gauss(x, a, x0, sigma):
   return a*np.exp(-(x-x0)**2/(2*(sigma**2)))

# set starting values for the Gaussian fit
par = [g_max,g_ave,g_std]

g_uncertainty = g_std/np.sqrt(i)
print("Gaussian parameters:",par)

# calculate the uncertainty on each bin as sqrt(entries)
y_err = []
x_val = []
y_val = []
for i in range(len(counts)):
    e = np.sqrt(counts[i])
    if e > 0:
        y_err.append(e)
        x_val.append((edges[i]+edges[i+1])/2)
        y_val.append(counts[i])

plt.errorbar(x_val,y_val,yerr=y_err,fmt='ko')
plt.xlabel(r'g[m/$s^2$]', fontsize = 15)
plt.ylabel(r'#of measurements per bin', fontsize = 15)

#using the starting parameters, fit a Gaussian curve to the data
popt, pcov = curve_fit(gauss, x_val, y_val,p0 = par, sigma = y_err)

# get parameter errors as sqrt of diagonal of covariance matrix
perr = np.sqrt(np.diag(pcov))

# calculate the chi^2
chi2 = 0.0
for i in range(len(x_val)):
    x = (gauss(x_val[i],*popt) - y_val[i])/y_err[i]
    chi2 = chi2 + x*x
    
ndof = len(x_val) - len(par)

print("Chi^2 = ",chi2,"for ",ndof,"#DOF")
    
print("Fitted parameters = ",popt)
print("Uncertainties = ",perr)

# plot Gaussian with fitted parameters
xx = np.arange(xmin,xmax,0.1)
plt.plot(xx, gauss(xx, *popt),'g',label="Gaussian fit")